# Xatbot d'assessorament d'habitatges amb LLM

 En aquest notebook farem una nova versió del chatbot recomanador d'habitatges, però en aquest cas fent servir un LLM i Tools.  

En primer lloc, indicarem a Google Colab com ha de tractar seqüències llargues de text. Això ens farà més senzill llegir els outputs del model.

IPython funciona com un lloc web i, per tant, podem modificar la seva aparença fent servir CSS. En aquest cas, farem servir l'opció pre-wrap per als espais al text. Aquesta opció fa que el text continuï a la següent línia en lloc de continuar per fora amb un desplaçament (scroll).

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

Ara hem de carregar el nostre LLM, que serà responsable de la part de llenguatge natural del sistema i a més cridarà les tools necessàries per trobar els habitatges correctes.

És important que trieu un model que estigui entrenat per fer servir tools (a vegades ho veureu com function calling). Per al nostre sistema hem triat Qwen2.5-Coder-3B-Instruct, que les suporta i a més és prou petit per poder-lo fer servir a Google Colab.

In [ ]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load the model and tokenizer (e.g., Qwen2.5-Coder-3B-Instruct)
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## Tools

Les tools són funcions de codi que el nostre LLM pot decidir cridar per fer tasques concretes. Per exemple, els LLMs tendeixen a tenir errors amb operacions matemàtiques. És molt més fiable crear una tool que faci les operacions directament, en lloc d'entrenar el model per arreglar els seus errors.

Definim una funció per escollir les cases que compleixen els requisits dels usuaris, anomenada "filter_real_state", que rep com a argument les preferències de l'usuari (localització, preu, habitacions, etc.) i retorna una llista d'habitatges dintre del rang correcte.

In [ ]:
# ----------------------------------------------------
# Custom Tool Backend Logic
# ----------------------------------------------------
def filter_real_estate(type=None, location=None, max_price=None, min_bedrooms=None, elevator=None, file_path='/content/house_data.json'):
    """Parses and filters your specific house real estate dataset."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            db = json.load(f)

        # Target the nested 'houses' array directly
        houses = db.get("houses", [])
        filtered_houses = []

        def parse_price(price_str):
            """Helper to normalize prices like '250k' or '1200' to integers"""
            price_str = str(price_str).lower().strip()
            if 'k' in price_str:
                return int(float(price_str.replace('k', '')) * 1000)
            return int(price_str)

        for house in houses:
            # 1. Filter by listing type (rent/sale)
            if type and house.get("type") != type:
                continue

            # 2. Filter by location (case-insensitive substring match)
            if location and location.lower() not in house.get("location", "").lower():
                continue

            # 3. Filter by minimum bedrooms
            if min_bedrooms and int(house.get("bedrooms", 0)) < int(min_bedrooms):
                continue

            # 4. Filter by elevator presence
            if elevator and house.get("elevator") != elevator:
                continue

            # 5. Filter by maximum price
            if max_price:
                house_price = parse_price(house.get("price", 0))
                if house_price > int(max_price):
                    continue

            filtered_houses.append(house)

        return json.dumps(filtered_houses, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})

Perquè el nostre LLM pugui decidir quan ha de cridar una tool, hem de definir al nostre prompt la nostra tool i com s'ha d'utilitzar.

El format més utilitzat és un JSON on definim els camps següents:

- **name**: El nom de la funció tal com el model l'ha de cridar.

- **description**: Descripció de per a què serveix la funció. Ha d'incloure tota la informació necessària i ser clara. El model prendrà la decisió de quina tool ha de cridar basat en aquesta informació.

 - **parameters**: Tots els paràmetres que espera la funció, indicant el seu tipus, opcions acceptades i una descripció del que representen. Igual que amb la funció, aquesta descripció és el que permet al LLM escollir quin valor ha d'assignar a cada paràmetre.

A continuació teniu un exemple de com hem definit la nostra funció filter_real_state:

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "filter_real_estate",
            "description": "Filters a real estate database for houses based on criteria like type, location, price, and size.",
            "parameters": {
                "type": "object",
                "properties": {
                    "type": {
                        "type": "string",
                        "enum": ["rent", "sale"],
                        "description": "Whether the user wants to rent or buy a house."
                    },
                    "location": {
                        "type": "string",
                        "description": "The city or neighborhood name (e.g., 'Barcelona', 'Santa Coloma de Gramenet')."
                    },
                    "max_price": {
                        "type": "integer",
                        "description": "The maximum price limit. For 'rent', pass the direct value (e.g., 1500). For 'sale', treat '250k' as 250000, so pass 250000."
                    },
                    "min_bedrooms": {
                        "type": "integer",
                        "description": "The minimum number of bedrooms required."
                    },
                    "elevator": {
                        "type": "string",
                        "enum": ["Yes", "No"],
                        "description": "Whether an elevator is strictly required."
                    }
                },
                "required": [] # All arguments are optional to allow partial filtering
            }
        }
    }
]

Anem a provar el nostre sistema amb un exemple:
Un usuari fa servir el nostre LLM i li diu la següent frase:

**"I want to buy a house in Barcelona with at least 3 bedrooms and a maximum budget of 400,000 euros."**

Com responem la seva pregunta?

En primer lloc, hem de definir el nostre prompt. Recordeu que estarà compost de dues parts principals:

- **System Prompt:** Instruccions que defineixen el comportament general del sistema (to, limitacions, etc.). Venen donades pels responsables del sistema.

- **Task Prompt:** Tasca concreta que demana l'usuari. En aquest cas, recomanar una casa d'acord amb unes preferències.

Aquí teniu un exemple per al nostre sistema:

In [ ]:
# 1. System Prompt and Initial Chat History in English
# This explicitly instructs the model to reason and respond in English.
conversation_history = [
    {
        "role": "system",
        "content": (
            "You are an expert real estate assistant. Your goal is to help the user find a house.\n"
            "TOOL RULES:\n"
            "1. If you need to search for houses, generate a ```json code block containing the tool call parameters.\n"
            "2. When you receive the output from the 'tool' role, DO NOT call the tool again with the same parameters.\n"
            "3. As soon as you have the JSON data from the tool, process the information and respond to the user directly in natural language. Do not generate any more JSON markdown code blocks!"
        )
    },
    {
        "role": "user",
        "content": "I want to buy a house in Barcelona with at least 3 bedrooms and a maximum budget of 400,000 euros."
    }
]

Definim un mètode que ens permeti extreure la trucada a la tool de la sortida del LLM i cridar la funció corresponent.

En aquest cas hem tingut en compte un problema extra. Els models (sobretot si són petits) poden entrar en un loop de demanar la mateixa informació sense parar. Limitem el nombre màxim de trucades a 2 i no es pot repetir dos cops la mateixa.



In [ ]:
import json
import re

# Map function names to executable Python functions
AVAILABLE_TOOLS = {
    "filter_real_estate": filter_real_estate
}


def call_tool(messages, function_name, arguments, clean_assistant_content, verbose=True):

    if verbose:
        print(f"\n[🔧 Executing Tool]: {function_name} with parameters: {arguments}")

    if function_name in AVAILABLE_TOOLS:
        tool_output = AVAILABLE_TOOLS[function_name](**arguments)

        # Safeguard string conversion if the filtering response array arrives empty
        if tool_output == "[]" or not tool_output:
            tool_output = "No houses matching all your strict filtering criteria were found in the database."

        if verbose:
            print(f"[📥 Tool Output Payload]: {tool_output}")

        # Append exact historical state records to chat memory
        messages.append({"role": "assistant", "content": clean_assistant_content})
        messages.append({"role": "tool", "name": function_name, "content": str(tool_output)})
    return messages

Ara farem la funció per cridar el nostre model. En fer servir tools, no podem cridar el model i donar la resposta directament a l'usuari, perquè veuria la crida a la tool en lloc de la resposta al seu text.

Definim la funció query_model_with_tools, que primer crida el model amb la informació que proporciona l'usuari. Tot seguit, fem servir una expressió regular per detectar si hi tenim una crida a una tool o no. Si no hi ha cap crida, podem donar resposta a l'usuari. Si hi ha una crida, hem de cridar la nostra tool, afegir el seu resultat al context del model i tornar-lo a cridar per generar una nova resposta. En alguns casos, això pot necessitar més d'una crida o fer servir més d'una tool per arribar a la resposta.


In [ ]:
def query_model_with_tools(messages, verbose=True, current_iteration=1, max_iterations=5):
    if current_iteration > max_iterations:
        print(f"⚠️ Maximum iteration limit reached ({max_iterations}). Forcing termination.")
        return "Sorry, I was unable to complete your request within the maximum allowed steps."

    if verbose:
        print(f"\n================ [Iteration {current_iteration}] ================")

    # Apply chat template and enforce clean sequence generation
    text = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Encode stop tokens cleanly
    im_end_id = tokenizer.encode("<|im_end|>")[0]

    # Generate tokens with low temperature configuration for reliable agent behavior
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.1,         # Low temperature prevents loop hallucinations
        top_p=0.9,
        do_sample=False,         # Deterministic generation ensures structured rule following
        eos_token_id=[im_end_id, tokenizer.eos_token_id],
        pad_token_id=tokenizer.eos_token_id
    )

    # Clean array slicing to isolate newly generated tokens
    new_tokens = generated_ids[0][len(model_inputs.input_ids[0]):]
    decoded_message = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    raw_message = tokenizer.decode(new_tokens, skip_special_tokens=False).strip()

    # Clear structural trailing boundary tags
    clean_assistant_content = raw_message.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()

    if verbose:
        print("\n--- Model Output ---")

    # Detect tool call block requests via regex
    json_block_match = re.search(r"```json\s*(.*?)\s*```", decoded_message, re.DOTALL)

    if json_block_match:
        tool_call_payload = json_block_match.group(1)
        tool_call_json = json.loads(tool_call_payload)
        function_name = tool_call_json.get("name")
        arguments = tool_call_json.get("arguments", {})

        try:

            # REPETITION LOOP CONTROL:
            # Intercepts the process if the model repeats an identical tool call execution signature
            if len(messages) >= 2 and messages[-2].get("role") == "assistant" and function_name in messages[-2].get("content", ""):
                messages.append({
                    "role": "user",
                    "content": "You have already executed that tool and have the results above. Please directly answer the user using the available data right now."
                })
            else:
                messages = call_tool(messages, function_name, arguments, clean_assistant_content, verbose)
            # Recurse into the next response turn
            return query_model_with_tools(messages, verbose, current_iteration + 1, max_iterations)

        except json.JSONDecodeError:
            messages.append({"role": "assistant", "content": clean_assistant_content})
            messages.append({"role": "user", "content": "Error: Your markdown output is not valid JSON. Please correct it and answer the user."})
            return query_model_with_tools(messages, verbose, current_iteration + 1, max_iterations)

    # Base Case: Direct text message layout response
    return clean_assistant_content



Provem el nostre sistema. Amb l'opció verbose=True podeu veure totes les crides intermèdies. Amb verbose=False podeu veure només la resposta a l'usuari.

In [ ]:
print(query_model_with_tools(conversation_history, verbose=True))

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



================ [Iteration 1] ================

--- Model Output ---

[🔧 Executing Tool]: filter_real_estate with parameters: {'type': 'sale', 'location': 'Barcelona', 'max_price': 400000, 'min_bedrooms': 3, 'elevator': 'No'}
[📥 Tool Output Payload]: [
  {
    "id": 20,
    "type": "sale",
    "bedrooms": "3",
    "bathrooms": "2",
    "price": "360k",
    "square_meters": "90",
    "floor": "2",
    "elevator": "No",
    "commercial_use": "No",
    "terrace": "Yes",
    "location": "Barcelona"
  }
]

================ [Iteration 2] ================

--- Model Output ---
Based on your search criteria, there is one house available in Barcelona that meets your requirements:

- Bedrooms: 3
- Bathrooms: 2
- Price: 360,000 euros
- Square meters: 90
- Floor: 2
- Elevator: No
- Commercial use: No
- Terrace: Yes
- Location: Barcelona

This property is a good fit for your budget and space needs. If you have any other questions or need further assistance, feel free to ask!


# Xatbot Complert

Ara que ja tenim un LLM capaç de respondre una pregunta sobre la nostra base de dades, podem crear el nostre xatbot.

Per fer-ho, hem creat una nova funció start_chat_session, la qual, donat un indicatiu de sistema (system prompt), inicia una conversa afegint els torns anteriors al context perquè el model tingui tota la informació necessària.

In [ ]:
def start_chat_session(session_history):
    """
    Manages a continuous, multi-turn chat session with the user.
    Maintains the full conversation history across multiple independent queries.
    """
    print("====================================================")
    print("🏠 House Buying Assistant Initialized! Type 'exit' or 'quit' to end the chat.")
    print("====================================================")

    while True:
        # 1. Capturar la entrada del usuario
        try:
            user_input = input("\nYou: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nGoodbye!")
            break

        if user_input.lower() in ['exit', 'quit']:
            print("Thank you for using the House Buying Assistant. Goodbye!")
            break

        if not user_input:
            continue

        session_history.append({"role": "user", "content": user_input})
        final_response = query_model_with_tools(session_history, verbose=False)

        if session_history[-1]["role"] != "assistant":
            session_history.append({"role": "assistant", "content": final_response})

        print(f"\nAssistant: {final_response}")




Proveu a demanar una casa al nostre xatbot ara amb més d'un torn de conversa. Per finalitzar, podeu escriure quit o exit a la finestra de text.

In [ ]:
  session_history = [
      {
          "role": "system",
          "content": (
              "You are an expert real estate assistant. Your goal is to help the user find a house.\n"
              "TOOL RULES:\n"
              "1. If you need to search for houses, generate a ```json code block containing the tool call parameters.\n"
              "2. When you receive the output from the 'tool' role, DO NOT call the tool again with the same parameters.\n"
              "3. As soon as you have the JSON data from the tool, process the information and respond to the user directly in natural language. Do not generate any more JSON markdown code blocks!"
          )
      }
  ]

  start_chat_session(session_history)

Sembla que el nostre xatbot és prou bo per recomanar cases, però serà prou bo per interactuar amb usuaris reals?

Com vam dir a classe, testejar atacs contra el nostre sistema (red teaming) és molt important per assegurar la qualitat de les respostes que donem als nostres usuaris.

 <h1><a name="ex-one"><center> ✏ Exercici 1 ✏</a></h1>


Manteniu una conversa més llarga amb el xatbot. Quines situacions donen errors? Com podríem solucionar aquests problemes fent servir guardrails?

Pista: Demaneu al xatbot un pis amb terrassa.

In [ ]:
start_chat_session(session_history)

 <h1><a name="ex-one"><center> ✏ Exercici 2 ✏</a></h1>

Un altre atac popular és la injecció de prompt (prompt injection), on l'usuari intenta aconseguir un comportament no desitjat del sistema a partir del seu prompt.

Intenteu fer prompt injection al nostre xatbot. Com es comporta el model? Com podríem evitar aquest comportament?

In [ ]:
start_chat_session(session_history)

🏠 House Buying Assistant Initialized! Type 'exit' or 'quit' to end the chat.

Goodbye!


 <h1><a name="ex-one"><center> ✏ Entregable ✏</a></h1>
Milloreu el xatbot actual arreglant els problemes que hem detectat. Els punts que heu de millorar són:

- Millorar el flux de la conversa, especialment com comença i com acaba amb l'usuari.

- Millorar la nostra tool per filtrar els habitatges. Ha de tenir en compte tots els arguments importants. Us podeu basar en la funció que vau dissenyar per al sistema de regles de la primera part.

- Afegir guardrails al sistema per evitar comportaments no desitjats. Podeu consultar a les diapositives de teoria els tipus de guardrails i els llocs on implementar-los.
